In [1]:
import xarray as xr
import earthaccess
import numpy as np
from pyresample import image, geometry, kd_tree
import h5py
import matplotlib.pyplot as plt
from tqdm import tqdm

In [2]:
earthaccess.login()

Enter your Earthdata Login username:  kandreadis
Enter your Earthdata password:  ········


## Enhanced SMAP 9-km

In [3]:
results = earthaccess.search_data(short_name='SPL3SMP_E', 
                                  bounding_box=(164.7,-47.8,179.6,-32.8), 
                                  temporal=("2022-1-1", "2024-12-31"), 
                                  cloud_hosted=True)

/home/ubuntu/smap/.venv/lib/python3.14/site-packages/earthaccess/results.py:348: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


In [4]:
files = earthaccess.open(results)

/home/ubuntu/smap/.venv/lib/python3.14/site-packages/earthaccess/store.py:523: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum([granule.size() for granule in granules]) / 1024, 2)


QUEUEING TASKS | :   0%|          | 0/1050 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1050 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1050 [00:00<?, ?it/s]

In [5]:
def get_gridded_smap9(filename):
    with h5py.File(filename) as f:
        sm = f["Soil_Moisture_Retrieval_Data_AM"]["soil_moisture"][:]
        lats = f["Soil_Moisture_Retrieval_Data_AM"]["latitude"][:]
        lats[lats == -9999] = np.nan
        lons = f["Soil_Moisture_Retrieval_Data_AM"]["longitude"][:]
        lons[lons == -9999] = np.nan
    swath_def = geometry.SwathDefinition(lons=lons, lats=lats)
    lon9, lat9 = np.meshgrid(np.arange(164.7, 179.6, 0.09), np.arange(-32.8, -47.8, -0.09))
    area_def = geometry.GridDefinition(lons=lon9, lats=lat9)
    result = kd_tree.resample_nearest(swath_def, sm, area_def, radius_of_influence=9000, epsilon=0.5)
    result[result <= 0] = np.nan
    return xr.DataArray(result, dims=('lat', 'lon'), coords=dict(lat=lat9[:, 0], lon=lon9[0, :]))

In [6]:
for filename in tqdm(files[503:]):
    sm = get_gridded_smap9(filename)
    outfile = filename.full_name.split("/")[-1].replace("h5", "nc")
    sm.to_netcdf(f"data/smap9km/{outfile}")

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 547/547 [23:46<00:00,  2.61s/it]


## SMAP 36-km

In [3]:
results = earthaccess.search_data(short_name='SPL3SMP', 
                                  bounding_box=(164.7,-47.8,179.6,-32.8), 
                                  temporal=("2022-1-1", "2024-12-31"), 
                                  cloud_hosted=True)

/home/ubuntu/smap/.venv/lib/python3.14/site-packages/earthaccess/results.py:348: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


In [4]:
files = earthaccess.open(results)

/home/ubuntu/smap/.venv/lib/python3.14/site-packages/earthaccess/store.py:523: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum([granule.size() for granule in granules]) / 1024, 2)


QUEUEING TASKS | :   0%|          | 0/1050 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1050 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1050 [00:00<?, ?it/s]

In [5]:
def get_gridded_smap36(filename):
    with h5py.File(filename) as f:
        sm = f["Soil_Moisture_Retrieval_Data_AM"]["soil_moisture"][:]
        lats = f["Soil_Moisture_Retrieval_Data_AM"]["latitude"][:]
        lats[lats == -9999] = np.nan
        lons = f["Soil_Moisture_Retrieval_Data_AM"]["longitude"][:]
        lons[lons == -9999] = np.nan
    swath_def = geometry.SwathDefinition(lons=lons, lats=lats)
    long, latg = np.meshgrid(np.arange(164.7, 179.6, 0.36), np.arange(-32.8, -47.8, -0.36))
    area_def = geometry.GridDefinition(lons=long, lats=latg)
    result = kd_tree.resample_nearest(swath_def, sm, area_def, radius_of_influence=36000, epsilon=0.5)
    result[result <= 0] = np.nan
    return xr.DataArray(result, dims=('lat', 'lon'), coords=dict(lat=latg[:, 0], lon=long[0, :]))

In [6]:
for filename in tqdm(files[787:]):
    sm = get_gridded_smap36(filename)
    outfile = filename.full_name.split("/")[-1].replace("h5", "nc")
    sm.to_netcdf(f"data/smap36km/{outfile}")

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 263/263 [03:09<00:00,  1.38it/s]


## SMAP/Sentinel

In [ ]:
results = earthaccess.search_data(short_name='SPL2SMAP_S', 
                                  bounding_box=(164.7,-47.8,179.6,-32.8), 
                                  temporal=("2016-1-1", "2024-12-31"), 
                                  cloud_hosted=True)

In [ ]:
files = earthaccess.open(results)

In [ ]:
def get_gridded_smap_sentinel(filename, res):
    with h5py.File(filename) as f:
        sm = f[f"Soil_Moisture_Retrieval_Data_{res}km"][f"soil_moisture_{res}km"][:]
        lats = f[f"Soil_Moisture_Retrieval_Data_{res}km"][f"latitude_{res}km"][:]
        lats[lats == -9999] = np.nan
        lons = f[f"Soil_Moisture_Retrieval_Data_{res}km"][f"longitude_{res}km"][:]
        lons[lons == -9999] = np.nan
    swath_def = geometry.SwathDefinition(lons=lons, lats=lats)
    long, latg = np.meshgrid(np.arange(164.7, 179.6, 0.01*res), np.arange(-32.8, -47.8, -0.01*res))
    area_def = geometry.GridDefinition(lons=long, lats=latg)
    result = kd_tree.resample_nearest(swath_def, sm, area_def, radius_of_influence=(res*1000), epsilon=0.5)
    result[result <= 0] = np.nan
    return xr.DataArray(result, dims=('lat', 'lon'), coords=dict(lat=latg[:, 0], lon=long[0, :]))

In [ ]:
for filename in tqdm(files[7442:]):
    sm = get_gridded_smap_sentinel(filename, 3)
    outfile = filename.full_name.split("/")[-1].replace("h5", "nc")
    sm.to_netcdf(f"data/smap3km/{outfile}")
    sm = get_gridded_smap_sentinel(filename, 1)
    outfile = filename.full_name.split("/")[-1].replace("h5", "nc")
    sm.to_netcdf(f"data/smap1km/{outfile}")